# G: S4-UPIR NoRelative · A100 · Fresh30

Upload whole **AnchorFlow_S4_UPIR_NoRelative_30ep_A100** to MyDrive, A100 → Run all.

U-PIR architecture IDENTICAL to F:4coarseNFE, no fine NODE/CNN,41input PIR,583880params.
Only training treatment changed: **relative_enabled=false, relative_weight=0**.
No DA3MONO archive/cache/pair computation needed. Metric teacher stays enabled.
Same seed42, fresh30, ImageNet encoder,1600/400, BF16train, FP32TRT+CUDA Graph inference.
New run/CODE/cache names prevent accidental resume of dual-teacher checkpoints.

Full400 benchmark +1000test/10pairs + historical comparison/acceptance report.
6cm parity unchanged; no speed/accuracy improvement guaranteed.
Historical A-E have relative KD ON; not identical-loss controls for this run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os,sys,json,hashlib,shutil,subprocess
from pathlib import Path
os.chdir('/content')  # uploads must NEVER land in frozen source
BUNDLE=Path('/content/drive/MyDrive/AnchorFlow_S4_UPIR_NoRelative_30ep_A100')
CODE=Path('/content/anchorflow_s4_upir_norel_code')
WORK=Path('/content/anchorflow_s4_upir_norel_work')
RUN_TAG='pair01'  # keep SAME tag/source/config for strict resume
assert BUNDLE.is_dir(),f'Upload WHOLE extracted folder to {BUNDLE}'


In [ ]:
manifest=json.loads((BUNDLE/'bundle_manifest.json').read_text())
for name,sha in manifest['files'].items():
    if name.endswith('.ipynb'):continue  # Colab changes notebook metadata/outputs
    src=BUNDLE/name
    assert src.is_file() and hashlib.sha256(src.read_bytes()).hexdigest()==sha,name
for name in [*manifest['files'],'bundle_manifest.json']:
    if name.endswith('.ipynb'):continue
    dest=CODE/name;dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.exists():
        assert dest.read_bytes()==(BUNDLE/name).read_bytes(),f'Local source changed: {name}; use clean runtime/new run for changes'
    else:shutil.copy2(BUNDLE/name,dest)
expected_py={name for name in manifest['files'] if name.endswith('.py')}
actual_py={p.relative_to(CODE).as_posix() for p in CODE.rglob('*.py') if '__pycache__' not in p.parts}
assert actual_py==expected_py,f'Unexpected/missing source files: {actual_py^expected_py}'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements-trt.txt')],check=True)
import torch,tensorrt
assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0),'Select A100 runtime'
assert torch.cuda.is_bf16_supported(including_emulation=False),'Native BF16 required'
assert tensorrt.__version__.startswith('10.13.3'),tensorrt.__version__
print(torch.__version__,torch.cuda.get_device_name(0),'TensorRT',tensorrt.__version__)
test_result=subprocess.run([sys.executable,'-u','-m','unittest','discover','-s',str(CODE),'-p','test_contracts.py','-v'],cwd='/content',stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print(test_result.stdout,flush=True)
assert test_result.returncode==0,'Unit test failed; inspect FULL output above; do not skip gates.'


In [ ]:
cfg=json.loads((CODE/'config.json').read_text())
cfg.update(work=str(WORK),run_name='S4_UPIR_NoRelative_Fresh30_'+RUN_TAG)
RUN=Path(cfg['drive_runs'])/cfg['run_name'];RUN.mkdir(parents=True,exist_ok=True)
CONFIG=RUN/'resolved_config.json'
if CONFIG.exists():
    assert json.loads(CONFIG.read_text())==cfg,'Recipe changed: use NEW RUN_TAG'
else:CONFIG.write_text(json.dumps(cfg,indent=2))
VARIANT='G_s4_upir_norel'
def command(action):
    log=RUN/f'{VARIANT}_{action}_console.log'
    args=[sys.executable,'-u',str(CODE/'run.py'),action,'--config',str(CONFIG),'--variant',VARIANT]
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(args,cwd='/content',stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);handle.write(line);handle.flush()
        rc=process.wait()
    if rc:raise RuntimeError(f'{action} failed; inspect {log}. No backend/gate fallback.')
print('Outputs:',RUN)

assert cfg['relative_enabled'] is False and cfg['relative_weight']==0


## 1. Prepare metric-only data and smoke

Required GeoLift_Data/teacher_subset_2000: selected_2000_ids.json,
kitti_trainval_2000.tar, metric_coarse_train_2000.tar. **Relative archive is NOT
required/read**. Test uses GeoLift_Data/test_1000/kitti_test_1000.tar or the
original canonical download. Metric teacher remains train-only.
Smoke checks backward and untrained FP32 TensorRT graph BEFORE long training.

In [ ]:
ready=all((WORK/name).is_file() for name in ('bundle_ready.json','metric_train/teacher_report.json','test_ready.json'))
need=4 if ready else 30
assert shutil.disk_usage('/content').free>need*2**30,f'Need >{need} GiB local SSD'
command('prepare')
shutil.copy2(WORK/'data_contract.json',RUN/'data_contract.json')
command('smoke')
smoke=json.loads((RUN/VARIANT/'smoke_report.json').read_text())
assert smoke['passed'] and smoke['coarse']['nfe']==4 and smoke['fine']['nfe']==0
display(smoke)
assert smoke['upir']['input_channels']==41 and smoke['upir']['new_weight_abs_max']==0


## 2. Fresh30 / epoch-boundary resume

Same seed42, LR/optimizer/schedule/loss coefficients EXCEPT relative KD OFF.
Metric teacher KD stays .012→.006, no compensating loss increase. No early stop.
Same NEW RUN_TAG/source/config resumes this metric-only run's last.pth; do NOT
reuse dual-teacher checkpoints. All relative raw/weighted/ramp/coverage logs must be0.

In [ ]:
command('train')
status=json.loads((RUN/VARIANT/'training_status.json').read_text())
assert status['completed_epochs']==30 and status['status']=='complete'
display(status)


## 3. FP32 TensorRT + CUDA Graph benchmark and test predictions

Full400,25,424,992GT pixels; nativeBF16 + FP32 references. A/B/A freshness
and fixed mean1mm/max6cm pixel gates; global drift2mm RMSE/.02iRMSE.
No silent gate relaxation or PyTorch fallback. Test1000 +10 previews.
Benchmarking failure does NOT require retraining: rerun only this cell after diagnosis.

In [ ]:
command('benchmark')
report=json.loads((RUN/VARIANT/'trt_benchmark.json').read_text())
assert report['qualification_passed']
display(report['trt_graph']['all'])
display(report['timing']['trt_graph'])


## 4. Historical comparison and acceptance report

Old A/B/C/D/E are NOT retrained or remeasured. Qualified Drive reports/checkpoints
are verified for data, common init and recipe; if absent, snapshots are labeled
unverified. A-D have30epochs like F; E26/MSG are not training-budget matched.
Historical runtime is not same-session speed proof. Small wins need repeated
same-session timing / additional seeds. Numerical parity success does not mean
accuracy/latency goals were achieved: inspect acceptance.json, retain D on failure.

Relative treatment differs from historical A-E; table includes relative_kd on/off.
Split/metric-cache/init checks remain; only relative cache metadata is excluded
from the old-vs-new data contract comparison, not strict checkpoint resume.

In [ ]:
command('compare')
import pandas as pd
table=pd.read_csv(RUN/'comparison.csv')
display(table[['variant','rmse_m','irmse_km_inv','mae_m','boundary3_rmse_m','median_ms','p95_ms','fps','latency_reduction_vs_A_pct','matched_training_verified']])
print('1000 test predictions +10 RGB/depth pairs:',RUN/VARIANT)
print('Full-resolution RGB/A/B/C/D/E/F comparisons:',RUN/'comparison_previews')
print('Results:',RUN/'comparison.csv')

display(json.loads((RUN/'acceptance.json').read_text()))
